# Evaluacion de enfoques de vision para GestCorp 98

Comparamos distintos enfoques para localizar 7 elementos de la interfaz sobre las 3 capturas de `experiments/ground_truth/`, contra el ground truth real (`alta_cliente.json`): un circulo verde marca la posicion real y uno rojo la prediccion de cada enfoque, en imagenes guardadas dentro de `vision_evaluation/<enfoque>/`.

Orden: `qwen2.5vl:7b` (Ollama, ya descargado) -> RapidOCR (sin LLM) -> OmniParser (YOLOv8 + Florence-2, busqueda de texto sobre el caption, sin LLM) -> Molmo -> UI-TARS (estos dos ultimos via Hugging Face `transformers`, pesan ~16 GB cada uno, no ejecutados todavia).

`llama3.2-vision:11b` se probo y se descarto: falla al cargar en Ollama por un bug conocido y sin resolver ([issue #16547](https://github.com/ollama/ollama/issues/16547)), no algo de este notebook.

In [1]:
import base64
import time
from pathlib import Path

import httpx
import torch
from PIL import Image

GT_DIR = Path("ground_truth")

# (imagen, nombre_del_target, descripcion para el prompt)
TARGETS = [
    ("01_login.png", "usuario_field", "el campo de texto donde se escribe el nombre de usuario (ya contiene 'admin')"),
    ("01_login.png", "contraseña_field", "el campo de texto vacio donde se escribe la contraseña"),
    ("01_login.png", "entrar_button", "el boton 'Entrar' para iniciar sesion"),
    ("02_alta_cliente.png", "nombre_field", "el campo de texto vacio para escribir el nombre del cliente, en el formulario 'Alta de cliente'"),
    ("02_alta_cliente.png", "nif_field", "el campo de texto vacio para escribir el NIF del cliente"),
    ("02_alta_cliente.png", "anadir_button", "el boton 'Añadir cliente' para guardar el nuevo cliente"),
    ("03_popup_validacion.png", "popup_aceptar_button", "el boton 'Aceptar' del popup de aviso 'Datos incompletos'"),
]

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA disponible: True
GPU: NVIDIA GeForce RTX 5090 Laptop GPU


## 1. qwen2.5vl:7b (Ollama)

Modelo de vision generalista, ya descargado en Ollama. Se le pide explicitamente el formato de grounding que usa Qwen (JSON con coordenadas).

**Prompt v2** (el primer intento dio mucho error, ver tabla mas abajo): ademas de la descripcion aislada del target, ahora se le da tambien el contexto completo de la pantalla (que aplicacion es, que hay en cada zona, que etiquetas hay y donde), para que no confunda, por ejemplo, las cabeceras de la tabla de clientes ("Nombre", "NIF") con las etiquetas del formulario "Alta de cliente" ("Nombre:", "NIF:"), que dicen casi lo mismo pero estan en sitios distintos de la imagen.

In [2]:
OLLAMA_URL = "http://localhost:11434/api/generate"

# Contexto de cada pantalla completa: ayuda al modelo a entender donde esta
# cada elemento dentro del layout, en vez de darle solo la descripcion
# aislada del target. Por ejemplo, en 02_alta_cliente.png la cabecera de la
# tabla de clientes dice "Nombre" y "NIF", y el formulario de mas abajo dice
# "Nombre:" y "NIF:" -- sin contexto, el modelo puede confundir una con otra.
SCREEN_CONTEXT = {
    "01_login.png": (
        "Es la pantalla de login de 'GestCorp 98', una aplicacion de escritorio "
        "estilo Windows 98. En el centro hay un formulario con dos filas: la "
        "primera fila tiene la etiqueta 'Usuario:' seguida, a su derecha, de "
        "un campo de texto que ya contiene el valor 'admin'; la segunda fila "
        "tiene la etiqueta 'Contraseña:' seguida, a su derecha, de un campo de "
        "texto vacio. Debajo de esas dos filas hay un boton 'Entrar'."
    ),
    "02_alta_cliente.png": (
        "Es la pestaña 'Clientes' de 'GestCorp 98'. Arriba hay una tabla con "
        "clientes ya registrados (columnas Nombre, NIF, Email, Telefono, "
        "Ciudad, Fecha alta -- esas son solo cabeceras de columna, NO el "
        "formulario). Debajo de la tabla hay un formulario con el titulo "
        "'Alta de cliente' y varios campos de texto VACIOS organizados en dos "
        "filas: la primera fila tiene las etiquetas 'Nombre:', 'NIF:' y "
        "'Email:', cada una seguida a su derecha de su campo vacio; la "
        "segunda fila tiene 'Telefono:' y 'Ciudad:' con sus respectivos "
        "campos vacios. A la derecha de esos campos, en esa misma zona del "
        "formulario, hay un boton 'Añadir cliente'."
    ),
    "03_popup_validacion.png": (
        "Es la misma pestaña 'Clientes' de 'GestCorp 98' (tabla de clientes "
        "arriba, formulario 'Alta de cliente' abajo), pero con un dialogo "
        "nativo de Windows superpuesto en el centro de la pantalla. Ese "
        "dialogo tiene el titulo 'Datos incompletos', el mensaje 'Nombre y "
        "NIF son obligatorios.' y, debajo del mensaje, un boton 'Aceptar'."
    ),
}


def _ollama_generate(model: str, image_path: Path, prompt: str) -> dict:
    image_b64 = base64.b64encode(image_path.read_bytes()).decode("ascii")
    t0 = time.perf_counter()
    response = httpx.post(
        OLLAMA_URL,
        json={"model": model, "prompt": prompt, "images": [image_b64], "stream": False},
        timeout=180,
    )
    elapsed = time.perf_counter() - t0
    response.raise_for_status()
    return {"raw": response.json()["response"], "elapsed_s": elapsed}


def ask_qwen25vl(image_name: str, target_desc: str) -> dict:
    image_path = GT_DIR / image_name
    width, height = Image.open(image_path).size
    contexto = SCREEN_CONTEXT[image_name]
    prompt = (
        f"Esta es una captura de pantalla de {width}x{height} pixeles "
        f"(ancho x alto), sin recortar ni escalar. {contexto} "
        "Dentro de esa captura completa, localiza exactamente este elemento: "
        f"{target_desc}. "
        "Da las coordenadas de su punto central en pixeles, usando el "
        f"sistema de coordenadas de la imagen ORIGINAL de {width}x{height} "
        "(el origen (0,0) es la esquina superior izquierda de la imagen). "
        "Responde UNICAMENTE con un JSON con este formato exacto, sin texto "
        'adicional: {"x": <entero>, "y": <entero>}'
    )
    return _ollama_generate("qwen2.5vl:7b", image_path, prompt)


In [3]:
import json
import re
from pathlib import Path as _Path

from PIL import ImageDraw
import pandas as pd

GT = json.loads((GT_DIR / "alta_cliente.json").read_text(encoding="utf-8"))

EVAL_OUT_DIR = _Path("vision_evaluation") / "qwen2.5vl"
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)


def parse_xy(raw: str):
    """Extrae un punto (x, y) de la respuesta cruda del modelo.

    Intenta primero su formato nativo bbox_2d (y toma el centro), luego
    el {"x":.., "y":..} que le pedimos explicitamente, y como ultimo
    recurso los dos primeros numeros que encuentre en el texto.
    """
    m = re.search(
        r'"bbox_2d"\s*:\s*\[\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)',
        raw,
    )
    if m:
        x1, y1, x2, y2 = map(float, m.groups())
        return (x1 + x2) / 2, (y1 + y2) / 2
    m = re.search(r'"x"\s*:\s*(-?[\d.]+)\s*,\s*"y"\s*:\s*(-?[\d.]+)', raw)
    if m:
        return float(m.group(1)), float(m.group(2))
    nums = re.findall(r"-?\d+\.?\d*", raw)
    if len(nums) >= 2:
        return float(nums[0]), float(nums[1])
    raise ValueError(f"no pude extraer coordenadas de: {raw!r}")


filas = []
for image_name, target_key, target_desc in TARGETS:
    gt_x, gt_y = GT[image_name][target_key]
    pred_x = pred_y = error_px = tiempo = None
    try:
        resultado = ask_qwen25vl(image_name, target_desc)
        tiempo = resultado["elapsed_s"]
        pred_x, pred_y = parse_xy(resultado["raw"])
        error_px = ((pred_x - gt_x) ** 2 + (pred_y - gt_y) ** 2) ** 0.5
        estado = "ok"
    except Exception as exc:
        estado = f"ERROR: {exc}"

    filas.append(
        {
            "imagen": image_name,
            "target": target_key,
            "gt_x": gt_x,
            "gt_y": gt_y,
            "pred_x": pred_x,
            "pred_y": pred_y,
            "error_px": error_px,
            "tiempo_s": tiempo,
            "estado": estado,
        }
    )
    print(
        f"{image_name}/{target_key}: {estado}  "
        f"gt=({gt_x},{gt_y}) pred=({pred_x},{pred_y}) "
        f"error={None if error_px is None else round(error_px, 1)}px "
        f"({tiempo if tiempo is None else round(tiempo, 1)}s)"
    )

    # circulo verde = posicion real, rojo = prediccion del modelo
    img = Image.open(GT_DIR / image_name).convert("RGB")
    draw = ImageDraw.Draw(img)
    r = 18
    draw.ellipse((gt_x - r, gt_y - r, gt_x + r, gt_y + r), outline=(0, 200, 0), width=5)
    if pred_x is not None:
        draw.ellipse((pred_x - r, pred_y - r, pred_x + r, pred_y + r), outline=(255, 0, 0), width=5)
    img.save(EVAL_OUT_DIR / f"{_Path(image_name).stem}__{target_key}.png")

df_qwen = pd.DataFrame(filas)
aciertos = (df_qwen["error_px"] < 20).sum()
print(f"\nResumen qwen2.5vl: {aciertos}/{len(df_qwen)} aciertos (error < 20px)")
print(f"error medio: {df_qwen['error_px'].mean():.1f}px")
print(f"tiempo medio: {df_qwen['tiempo_s'].mean():.1f}s")
print(f"capturas con circulos guardadas en: {EVAL_OUT_DIR}/")
df_qwen


01_login.png/usuario_field: ok  gt=(272,138) pred=(238.0,119.0) error=38.9px (10.6s)


01_login.png/contraseña_field: ok  gt=(272,188) pred=(238.0,163.0) error=42.2px (2.4s)


01_login.png/entrar_button: ok  gt=(208,274) pred=(185.0,238.0) error=42.7px (2.4s)


02_alta_cliente.png/nombre_field: ok  gt=(219,1399) pred=(180.0,1230.0) error=173.4px (6.1s)


02_alta_cliente.png/nif_field: ok  gt=(525,1399) pred=(350.0,1240.0) error=236.4px (2.4s)


02_alta_cliente.png/anadir_button: ok  gt=(475,1483) pred=(420.0,1268.0) error=221.9px (2.4s)


03_popup_validacion.png/popup_aceptar_button: ok  gt=(1397,888) pred=(1230.0,775.0) error=201.6px (6.1s)

Resumen qwen2.5vl: 0/7 aciertos (error < 20px)
error medio: 136.8px
tiempo medio: 4.6s
capturas con circulos guardadas en: vision_evaluation\qwen2.5vl/


,imagen,target,gt_x,gt_y,pred_x,pred_y,error_px,tiempo_s,estado
0,01_login.png,usuario_field,272,138,238.0,119.0,38.948684,10.641814,ok
1,01_login.png,contraseña_field,272,188,238.0,163.0,42.201896,2.369510,ok
2,01_login.png,entrar_button,208,274,185.0,238.0,42.720019,2.366143,ok
3,02_alta_cliente.png,nombre_field,219,1399,180.0,1230.0,173.441633,6.078735,ok
4,02_alta_cliente.png,nif_field,525,1399,350.0,1240.0,236.444497,2.431065,ok
5,02_alta_cliente.png,anadir_button,475,1483,420.0,1268.0,221.923410,2.377812,ok
6,03_popup_validacion.png,popup_aceptar_button,1397,888,1230.0,775.0,201.638290,6.134970,ok


### Prompt v3: coordenadas en porcentaje (0-100) en vez de pixeles

El prompt v2 (contexto completo de la pantalla) no bajo el error respecto al v1: la Y predicha es siempre ≈0.86 veces la Y real en los 7 targets, con v1 y v2 por igual -- la firma de un problema de **escalado**, no de comprension semantica. Probablemente la imagen se redimensiona en algun punto del pipeline (Ollama / el propio modelo) antes de "verla", y el modelo devuelve coordenadas en esa version reducida aunque el prompt diga "la imagen mide WxH".

Si eso es la causa, pedirle directamente un **porcentaje del ancho/alto** (0-100) en vez de pixeles absolutos deberia esquivar el problema: ya no depende de que resolucion uso internamente, solo de la posicion relativa dentro de la imagen (sea cual sea el tamaño con el que la vio). Convertimos ese porcentaje a pixeles reales nosotros mismos, con el ancho/alto real de la captura.

In [4]:
def ask_qwen25vl_pct(image_name: str, target_desc: str) -> dict:
    image_path = GT_DIR / image_name
    contexto = SCREEN_CONTEXT[image_name]
    prompt = (
        f"Esta es una captura de pantalla completa. {contexto} "
        "Dentro de esa captura, localiza exactamente este elemento: "
        f"{target_desc}. "
        "En vez de pixeles, da la posicion de su punto central como "
        "PORCENTAJE del ancho y del alto de la imagen: un numero entre 0 y "
        "100 en cada eje, donde x=0 es el borde izquierdo, x=100 es el borde "
        "derecho, y=0 es el borde superior e y=100 es el borde inferior. "
        "Responde UNICAMENTE con un JSON con este formato exacto, sin texto "
        'adicional: {"x_pct": <numero>, "y_pct": <numero>}'
    )
    return _ollama_generate("qwen2.5vl:7b", image_path, prompt)


def parse_pct(raw: str):
    """Extrae (x_pct, y_pct) de la respuesta cruda del modelo."""
    m = re.search(r'"x_pct"\s*:\s*(-?[\d.]+)\s*,\s*"y_pct"\s*:\s*(-?[\d.]+)', raw)
    if m:
        return float(m.group(1)), float(m.group(2))
    nums = re.findall(r"-?\d+\.?\d*", raw)
    if len(nums) >= 2:
        return float(nums[0]), float(nums[1])
    raise ValueError(f"no pude extraer porcentajes de: {raw!r}")


filas_pct = []
for image_name, target_key, target_desc in TARGETS:
    gt_x, gt_y = GT[image_name][target_key]
    width, height = Image.open(GT_DIR / image_name).size
    pred_x = pred_y = error_px = tiempo = x_pct = y_pct = None
    try:
        resultado = ask_qwen25vl_pct(image_name, target_desc)
        tiempo = resultado["elapsed_s"]
        x_pct, y_pct = parse_pct(resultado["raw"])
        # reescalamos el porcentaje que da el modelo a pixeles reales,
        # con el ancho/alto real de la captura (no el que el modelo
        # haya usado internamente para "verla")
        pred_x = x_pct / 100 * width
        pred_y = y_pct / 100 * height
        error_px = ((pred_x - gt_x) ** 2 + (pred_y - gt_y) ** 2) ** 0.5
        estado = "ok"
    except Exception as exc:
        estado = f"ERROR: {exc}"

    filas_pct.append(
        {
            "imagen": image_name,
            "target": target_key,
            "gt_x": gt_x,
            "gt_y": gt_y,
            "x_pct": x_pct,
            "y_pct": y_pct,
            "pred_x": pred_x,
            "pred_y": pred_y,
            "error_px": error_px,
            "tiempo_s": tiempo,
            "estado": estado,
        }
    )
    print(
        f"{image_name}/{target_key}: {estado}  "
        f"gt=({gt_x},{gt_y}) pct=({x_pct},{y_pct}) pred=({pred_x},{pred_y}) "
        f"error={None if error_px is None else round(error_px, 1)}px "
        f"({tiempo if tiempo is None else round(tiempo, 1)}s)"
    )

    # circulo verde = posicion real, rojo = prediccion del modelo (ya en pixeles)
    img = Image.open(GT_DIR / image_name).convert("RGB")
    draw = ImageDraw.Draw(img)
    r = 18
    draw.ellipse((gt_x - r, gt_y - r, gt_x + r, gt_y + r), outline=(0, 200, 0), width=5)
    if pred_x is not None:
        draw.ellipse((pred_x - r, pred_y - r, pred_x + r, pred_y + r), outline=(255, 0, 0), width=5)
    img.save(EVAL_OUT_DIR / f"{_Path(image_name).stem}__{target_key}__pct.png")

df_qwen_pct = pd.DataFrame(filas_pct)
aciertos = (df_qwen_pct["error_px"] < 20).sum()
print(f"\nResumen qwen2.5vl (prompt v3, porcentaje): {aciertos}/{len(df_qwen_pct)} aciertos (error < 20px)")
print(f"error medio: {df_qwen_pct['error_px'].mean():.1f}px")
print(f"error medio con prompt en pixeles (v1/v2): {df_qwen['error_px'].mean():.1f}px  <- para comparar")
print(f"tiempo medio: {df_qwen_pct['tiempo_s'].mean():.1f}s")
print(f"capturas con circulos guardadas en: {EVAL_OUT_DIR}/ (sufijo __pct)")
df_qwen_pct


01_login.png/usuario_field: ok  gt=(272,138) pct=(20.5,11.5) pred=(524.8,184.0) error=257.0px (2.5s)


01_login.png/contraseña_field: ok  gt=(272,188) pct=(235.0,162.0) pred=(6016.0,2592.0) error=6226.8px (2.4s)


01_login.png/entrar_button: ok  gt=(208,274) pct=(180.0,235.0) pred=(4608.0,3760.0) error=5613.6px (2.4s)


02_alta_cliente.png/nombre_field: ok  gt=(219,1399) pct=(15.0,90.0) pred=(384.0,1440.0) error=170.0px (2.5s)


02_alta_cliente.png/nif_field: ok  gt=(525,1399) pct=(205.0,905.0) pred=(5248.0,14480.000000000002) error=13907.5px (2.4s)


02_alta_cliente.png/anadir_button: ok  gt=(475,1483) pct=(450.0,950.0) pred=(11520.0,15200.0) error=17611.0px (2.4s)


03_popup_validacion.png/popup_aceptar_button: ok  gt=(1397,888) pct=(85.5,74.5) pred=(2188.8,1192.0) error=848.2px (2.5s)

Resumen qwen2.5vl (prompt v3, porcentaje): 0/7 aciertos (error < 20px)
error medio: 6376.3px
error medio con prompt en pixeles (v1/v2): 136.8px  <- para comparar
tiempo medio: 2.5s
capturas con circulos guardadas en: vision_evaluation\qwen2.5vl/ (sufijo __pct)


,imagen,target,gt_x,gt_y,x_pct,y_pct,pred_x,pred_y,error_px,tiempo_s,estado
0,01_login.png,usuario_field,272,138,20.5,11.5,524.8,184.0,256.951046,2.538586,ok
1,01_login.png,contraseña_field,272,188,235.0,162.0,6016.0,2592.0,6226.777015,2.437241,ok
2,01_login.png,entrar_button,208,274,180.0,235.0,4608.0,3760.0,5613.572481,2.408553,ok
3,02_alta_cliente.png,nombre_field,219,1399,15.0,90.0,384.0,1440.0,170.017646,2.483754,ok
4,02_alta_cliente.png,nif_field,525,1399,205.0,905.0,5248.0,14480.0,13907.526380,2.420990,ok
5,02_alta_cliente.png,anadir_button,475,1483,450.0,950.0,11520.0,15200.0,17611.022514,2.395452,ok
6,03_popup_validacion.png,popup_aceptar_button,1397,888,85.5,74.5,2188.8,1192.0,848.152840,2.539295,ok


**Resultado real: mucho peor, no mejor.** El modelo no respeta de forma fiable la instruccion de "porcentaje 0-100": en varios targets devuelve numeros fuera de ese rango (p.ej. `contraseña_field` -> `x_pct=235, y_pct=162`, `anadir_button` -> `x_pct=450, y_pct=950`) que son casi los mismos valores en bruto que ya daba el prompt en pixeles (v1/v2), como si ignorase la unidad pedida y siguiese emitiendo su formato de grounding nativo. Al dividir esos numeros "fuera de rango" entre 100 y multiplicar por el ancho/alto real, el error se dispara a miles de pixeles -- no porque el modelo "vea" peor, sino porque el formato de salida no es el que pedimos.

Solo en 2 de los 7 targets (`nombre_field`, `popup_aceptar_button`) el numero que devolvio esta dentro de 0-100 y es un porcentaje real, y aun asi el error no mejora frente al prompt en pixeles.

**Conclusion:** con la API simple de Ollama (`/api/generate`, sin su plantilla de chat ni tokens de vision propios), `qwen2.5vl:7b` esta anclado a su formato de grounding entrenado (coordenadas en una resolucion fija interna) y no cambia de comportamiento solo porque el prompt le pida otra unidad -- ni el contexto de pantalla (v2) ni pedir porcentaje (v3) lo corrigen. RapidOCR sigue siendo claramente la mejor opcion para los botones; para los campos vacios haria falta la heuristica de desplazamiento desde la etiqueta que ya se menciono en la seccion 2, en vez de insistir con qwen2.5vl por este camino.

## 2. RapidOCR (sin LLM)

OCR puro y determinista: detecta todo el texto de la imagen con su caja delimitadora. Primero mostramos la lista completa detectada en una imagen, y despues comparamos contra el ground truth igual que con qwen2.5vl: para los botones buscamos la deteccion cuyo texto coincide con la etiqueta del propio boton (su centro SI es el target real); para los campos vacios, al no haber texto dentro, usamos como aproximacion la etiqueta mas cercana (p.ej. "Usuario:" para `usuario_field`) sabiendo de antemano que su centro no sera el centro exacto del campo -- el circulo rojo frente al verde en cada imagen muestra ese desajuste.

In [5]:
from rapidocr_onnxruntime import RapidOCR

_ocr_engine = RapidOCR()


def ask_rapidocr(image_name: str, target_desc: str | None = None) -> dict:
    image_path = GT_DIR / image_name
    t0 = time.perf_counter()
    result, _ = _ocr_engine(str(image_path))
    elapsed = time.perf_counter() - t0
    detecciones = [
        {"texto": texto, "bbox": box, "confianza": round(float(score), 3)}
        for box, texto, score in (result or [])
    ]
    return {"raw": detecciones, "elapsed_s": elapsed}


In [ ]:
resultado = ask_rapidocr("02_alta_cliente.png")
print(f"tiempo: {resultado['elapsed_s']:.2f}s  -  {len(resultado['raw'])} detecciones\n")
for d in resultado["raw"]:
    print(d["texto"], "->", d["bbox"])


In [7]:
import unicodedata

EVAL_OUT_DIR_RAPIDOCR = _Path("vision_evaluation") / "rapidocr"
EVAL_OUT_DIR_RAPIDOCR.mkdir(parents=True, exist_ok=True)


def _normalize(texto: str) -> str:
    """minusculas y sin tildes/enies, para comparar sin depender de como
    el OCR haya transcrito acentos (p.ej. "contraseña" -> "contrasenia")."""
    sin_acentos = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    return sin_acentos.lower().strip()


# Para cada target, la palabra clave (normalizada) que debe contener el texto
# detectado por el OCR. En los botones el texto esta DENTRO del target
# (coincide con su propia etiqueta); en los campos vacios no hay nada que
# leer ahi, asi que usamos como aproximacion la etiqueta mas cercana (p.ej.
# "Usuario:"), sabiendo de antemano que su centro no sera el centro del
# campo real -- es justo lo que este cuadro va a dejar ver.
TARGET_OCR_KEYWORD = {
    "usuario_field": "usuario",
    "contraseña_field": "contras",
    "entrar_button": "entrar",
    "nombre_field": "nombre:",
    "nif_field": "nif:",
    "anadir_button": "anadir cliente",
    "popup_aceptar_button": "aceptar",
}


def match_ocr_target(detecciones: list[dict], keyword: str):
    """Busca la primera deteccion cuyo texto normalizado contiene `keyword`
    y devuelve el centro de su bbox, o None si no aparece ninguna."""
    for det in detecciones:
        if keyword in _normalize(det["texto"]):
            xs = [p[0] for p in det["bbox"]]
            ys = [p[1] for p in det["bbox"]]
            return sum(xs) / len(xs), sum(ys) / len(ys)
    return None


_rapidocr_cache: dict[str, dict] = {}

filas_rapidocr = []
for image_name, target_key, _desc in TARGETS:
    gt_x, gt_y = GT[image_name][target_key]

    if image_name not in _rapidocr_cache:
        _rapidocr_cache[image_name] = ask_rapidocr(image_name)
    resultado = _rapidocr_cache[image_name]
    tiempo = resultado["elapsed_s"]

    keyword = TARGET_OCR_KEYWORD[target_key]
    punto = match_ocr_target(resultado["raw"], keyword)
    pred_x = pred_y = error_px = None
    if punto is not None:
        pred_x, pred_y = punto
        error_px = ((pred_x - gt_x) ** 2 + (pred_y - gt_y) ** 2) ** 0.5
        estado = "ok"
    else:
        estado = f"ERROR: ninguna deteccion contiene '{keyword}'"

    filas_rapidocr.append(
        {
            "imagen": image_name,
            "target": target_key,
            "gt_x": gt_x,
            "gt_y": gt_y,
            "pred_x": pred_x,
            "pred_y": pred_y,
            "error_px": error_px,
            "tiempo_s": tiempo,
            "estado": estado,
        }
    )
    print(
        f"{image_name}/{target_key}: {estado}  "
        f"gt=({gt_x},{gt_y}) pred=({pred_x},{pred_y}) "
        f"error={None if error_px is None else round(error_px, 1)}px"
    )

    # circulo verde = posicion real, rojo = punto que da RapidOCR via etiqueta
    img = Image.open(GT_DIR / image_name).convert("RGB")
    draw = ImageDraw.Draw(img)
    r = 18
    draw.ellipse((gt_x - r, gt_y - r, gt_x + r, gt_y + r), outline=(0, 200, 0), width=5)
    if pred_x is not None:
        draw.ellipse((pred_x - r, pred_y - r, pred_x + r, pred_y + r), outline=(255, 0, 0), width=5)
    img.save(EVAL_OUT_DIR_RAPIDOCR / f"{_Path(image_name).stem}__{target_key}.png")

df_rapidocr = pd.DataFrame(filas_rapidocr)
aciertos = (df_rapidocr["error_px"] < 20).sum()
print(f"\nResumen rapidocr: {aciertos}/{len(df_rapidocr)} aciertos (error < 20px)")
print(f"error medio: {df_rapidocr['error_px'].mean():.1f}px")
print(f"tiempo medio (deteccion completa por imagen): {df_rapidocr['tiempo_s'].mean():.2f}s")
print(f"capturas con circulos guardadas en: {EVAL_OUT_DIR_RAPIDOCR}/")
df_rapidocr


01_login.png/usuario_field: ok  gt=(272,138) pred=(112.90322303771973,137.82051467895508) error=159.1px
01_login.png/contraseña_field: ok  gt=(272,188) pred=(98.06451225280762,188.4615478515625) error=173.9px
01_login.png/entrar_button: ok  gt=(208,274) pred=(209.03225708007812,275.00001525878906) error=1.4px


02_alta_cliente.png/nombre_field: ok  gt=(219,1399) pred=(60.64515924453735,1400.6410522460938) error=158.4px
02_alta_cliente.png/nif_field: ok  gt=(525,1399) pred=(387.0967559814453,1400.6410522460938) error=137.9px
02_alta_cliente.png/anadir_button: ok  gt=(475,1483) pred=(475.4838562011719,1485.8975219726562) error=2.9px


03_popup_validacion.png/popup_aceptar_button: ok  gt=(1397,888) pred=(1398.0645141601562,889.7436218261719) error=2.0px

Resumen rapidocr: 3/7 aciertos (error < 20px)
error medio: 90.8px
tiempo medio (deteccion completa por imagen): 2.30s
capturas con circulos guardadas en: vision_evaluation\rapidocr/


,imagen,target,gt_x,gt_y,pred_x,pred_y,error_px,tiempo_s,estado
0,01_login.png,usuario_field,272,138,112.903223,137.820515,159.096878,2.334270,ok
1,01_login.png,contraseña_field,272,188,98.064512,188.461548,173.936100,2.334270,ok
2,01_login.png,entrar_button,208,274,209.032257,275.000015,1.437214,2.334270,ok
3,02_alta_cliente.png,nombre_field,219,1399,60.645159,1400.641052,158.363344,2.062538,ok
4,02_alta_cliente.png,nif_field,525,1399,387.096756,1400.641052,137.913008,2.062538,ok
5,02_alta_cliente.png,anadir_button,475,1483,475.483856,1485.897522,2.937644,2.062538,ok
6,03_popup_validacion.png,popup_aceptar_button,1397,888,1398.064514,889.743622,2.042892,2.931032,ok


## 3. OmniParser (detector)

`microsoft/OmniParser-v2.0`: detector YOLOv8 de iconos (`ultralytics`) + Florence-2 como captioner (`transformers`). Pesos reales comprobados en Hugging Face (consultando metadatos, sin descargar nada): el repo completo son **~1.4 GB** (no los "2-3 GB" que decia antes esta nota), mas unos pocos MB del procesador de `microsoft/Florence-2-base` (solo tokenizer/config, no sus pesos -- esos no se descargan).

**Receta de carga verificada contra el `handler.py` oficial del repo** (no es tan directa como parece a primera vista): el detector sale de `icon_detect/model.pt`; el captioner son solo los PESOS afinados en `icon_caption/` (safetensors), pero el **procesador** (tokenizer + preprocesado de imagen) no viene en ese repo -- hay que cargarlo del modelo base original, `microsoft/Florence-2-base`. Los recortes de cada icono se redimensionan a 64x64 antes de pedir la descripcion (asi lo hace Microsoft en su propio handler).

Primero mostramos la lista completa de elementos detectados (caja + caption) en una imagen. Despues, igual que con RapidOCR, comparamos contra el ground truth buscando una palabra clave dentro del caption de cada elemento -- sin ningun LLM de por medio, la misma busqueda de texto simple que usamos con RapidOCR -- y usamos el centro de la caja elegida (que sale directamente del detector YOLOv8, en pixeles reales de la imagen) como prediccion.

⚠️ **No ejecutada por Claude, dejada lista para que la ejecutes tu.** Dispara la descarga de OmniParser (~1.4 GB) la primera vez.

In [8]:
_omniparser_detector = None
_omniparser_caption_model = None
_omniparser_caption_processor = None


def _load_omniparser():
    global _omniparser_detector, _omniparser_caption_model, _omniparser_caption_processor
    if _omniparser_detector is None:
        from huggingface_hub import snapshot_download
        from transformers import AutoModelForCausalLM, AutoProcessor
        from ultralytics import YOLO

        local_dir = snapshot_download("microsoft/OmniParser-v2.0")
        _omniparser_detector = YOLO(f"{local_dir}/icon_detect/model.pt")
        # el procesador se carga del Florence-2 base original: el repo de
        # OmniParser solo publica los pesos afinados del captioner, no su
        # propio procesador (asi lo hace su handler.py oficial)
        _omniparser_caption_processor = AutoProcessor.from_pretrained(
            "microsoft/Florence-2-base", trust_remote_code=True
        )
        _omniparser_caption_model = AutoModelForCausalLM.from_pretrained(
            f"{local_dir}/icon_caption", torch_dtype=torch.float16, trust_remote_code=True
        ).to("cuda")
    return _omniparser_detector, _omniparser_caption_model, _omniparser_caption_processor


def ask_omniparser_elements(image_name: str, target_desc: str | None = None) -> dict:
    detector, caption_model, caption_processor = _load_omniparser()
    image_path = GT_DIR / image_name
    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    t0 = time.perf_counter()
    results = detector(str(image_path), verbose=False)[0]

    crops = []
    cajas = []
    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        crop = image.crop((x1, y1, x2, y2)).resize((64, 64))
        crops.append(crop)
        cajas.append((x1, y1, x2, y2, float(box.conf[0])))

    captions = []
    if crops:
        inputs = caption_processor(
            images=crops, text=["<CAPTION>"] * len(crops), return_tensors="pt", do_resize=False
        ).to(caption_model.device, dtype=torch.float16)
        generated_ids = caption_model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=20,
            num_beams=1,
            do_sample=False,
        )
        captions = [
            texto.strip()
            for texto in caption_processor.batch_decode(generated_ids, skip_special_tokens=True)
        ]

    elementos = []
    for (x1, y1, x2, y2, conf), caption in zip(cajas, captions):
        elementos.append(
            {
                "bbox_px": [round(v, 1) for v in (x1, y1, x2, y2)],
                "bbox_norm": [
                    round(x1 / width, 4), round(y1 / height, 4),
                    round(x2 / width, 4), round(y2 / height, 4),
                ],
                "caption": caption,
                "confianza": round(conf, 3),
            }
        )
    elapsed = time.perf_counter() - t0
    return {"raw": elementos, "elapsed_s": elapsed}


In [ ]:
# prueba rapida: dispara la descarga de OmniParser (~1.4 GB) y muestra
# todos los elementos que detecta en la imagen del formulario
resultado = ask_omniparser_elements("02_alta_cliente.png")
print(f"tiempo: {resultado['elapsed_s']:.1f}s  -  {len(resultado['raw'])} elementos\n")
for el in resultado["raw"]:
    print(el)


In [10]:
# Visualizamos TODAS las cajas que detecta OmniParser en cada imagen (no
# solo la que coincide por palabra clave), para comprobar si localiza bien
# los campos/botones aunque el caption no diga el nombre exacto que
# buscamos (p.ej. el boton "Anadir cliente", que Florence-2 transcribe
# mal en el caption pero cuya caja sale casi perfecta).
from PIL import ImageFont

EVAL_OUT_DIR_OMNIPARSER = _Path("vision_evaluation") / "omniparser"
EVAL_OUT_DIR_OMNIPARSER.mkdir(parents=True, exist_ok=True)

_omniparser_cache: dict[str, dict] = {}
_font = ImageFont.load_default(size=28)

for image_name in dict.fromkeys(t[0] for t in TARGETS):
    if image_name not in _omniparser_cache:
        _omniparser_cache[image_name] = ask_omniparser_elements(image_name)
    elementos = _omniparser_cache[image_name]["raw"]

    img = Image.open(GT_DIR / image_name).convert("RGB")
    draw = ImageDraw.Draw(img)
    for i, el in enumerate(elementos):
        x1, y1, x2, y2 = el["bbox_px"]
        draw.rectangle((x1, y1, x2, y2), outline=(30, 120, 255), width=4)
        draw.text((x1 + 4, max(y1 - 32, 0)), str(i), fill=(30, 120, 255), font=_font)

    out_path = EVAL_OUT_DIR_OMNIPARSER / f"{_Path(image_name).stem}__todas_las_cajas.png"
    img.save(out_path)
    print(f"{image_name}: {len(elementos)} cajas dibujadas -> {out_path}")


01_login.png: 23 cajas dibujadas -> vision_evaluation\omniparser\01_login__todas_las_cajas.png


02_alta_cliente.png: 25 cajas dibujadas -> vision_evaluation\omniparser\02_alta_cliente__todas_las_cajas.png


03_popup_validacion.png: 26 cajas dibujadas -> vision_evaluation\omniparser\03_popup_validacion__todas_las_cajas.png


In [11]:
TARGET_OMNIPARSER_KEYWORD = {
    "usuario_field": "usuario",
    "contraseña_field": "contras",
    "entrar_button": "entrar",
    "nombre_field": "nombre",
    "nif_field": "nif",
    "anadir_button": "anadir",
    "popup_aceptar_button": "aceptar",
}


def match_omniparser_target(elementos: list[dict], keyword: str):
    """Busca el primer elemento cuyo caption normalizado (via `_normalize`,
    ya definida en la seccion de RapidOCR) contiene `keyword`, y devuelve
    el centro de su bbox_px, o None si no aparece ninguno."""
    for el in elementos:
        if keyword in _normalize(el["caption"]):
            x1, y1, x2, y2 = el["bbox_px"]
            return (x1 + x2) / 2, (y1 + y2) / 2
    return None


filas_omniparser = []
for image_name, target_key, _desc in TARGETS:
    gt_x, gt_y = GT[image_name][target_key]

    if image_name not in _omniparser_cache:
        _omniparser_cache[image_name] = ask_omniparser_elements(image_name)
    resultado = _omniparser_cache[image_name]
    elementos = resultado["raw"]
    tiempo = resultado["elapsed_s"]

    keyword = TARGET_OMNIPARSER_KEYWORD[target_key]
    punto = match_omniparser_target(elementos, keyword)
    pred_x = pred_y = error_px = None
    if punto is not None:
        pred_x, pred_y = punto
        error_px = ((pred_x - gt_x) ** 2 + (pred_y - gt_y) ** 2) ** 0.5
        estado = "ok"
    else:
        estado = f"ERROR: ningun caption contiene '{keyword}'"

    filas_omniparser.append(
        {
            "imagen": image_name,
            "target": target_key,
            "gt_x": gt_x,
            "gt_y": gt_y,
            "pred_x": pred_x,
            "pred_y": pred_y,
            "error_px": error_px,
            "tiempo_s": tiempo,
            "estado": estado,
        }
    )
    print(
        f"{image_name}/{target_key}: {estado}  "
        f"gt=({gt_x},{gt_y}) pred=({pred_x},{pred_y}) "
        f"error={None if error_px is None else round(error_px, 1)}px"
    )

    # circulo verde = posicion real, rojo = punto que da OmniParser via caption
    img = Image.open(GT_DIR / image_name).convert("RGB")
    draw = ImageDraw.Draw(img)
    r = 18
    draw.ellipse((gt_x - r, gt_y - r, gt_x + r, gt_y + r), outline=(0, 200, 0), width=5)
    if pred_x is not None:
        draw.ellipse((pred_x - r, pred_y - r, pred_x + r, pred_y + r), outline=(255, 0, 0), width=5)
    img.save(EVAL_OUT_DIR_OMNIPARSER / f"{_Path(image_name).stem}__{target_key}.png")

df_omniparser = pd.DataFrame(filas_omniparser)
aciertos = (df_omniparser["error_px"] < 20).sum()
print(f"\nResumen omniparser: {aciertos}/{len(df_omniparser)} aciertos (error < 20px)")
print(f"error medio: {df_omniparser['error_px'].mean():.1f}px")
print(f"tiempo medio (deteccion+caption por imagen): {df_omniparser['tiempo_s'].mean():.2f}s")
print(f"capturas con circulos guardadas en: {EVAL_OUT_DIR_OMNIPARSER}/")
df_omniparser


01_login.png/usuario_field: ERROR: ningun caption contiene 'usuario'  gt=(272,138) pred=(None,None) error=Nonepx
01_login.png/contraseña_field: ERROR: ningun caption contiene 'contras'  gt=(272,188) pred=(None,None) error=Nonepx
01_login.png/entrar_button: ERROR: ningun caption contiene 'entrar'  gt=(208,274) pred=(None,None) error=Nonepx
02_alta_cliente.png/nombre_field: ERROR: ningun caption contiene 'nombre'  gt=(219,1399) pred=(None,None) error=Nonepx
02_alta_cliente.png/nif_field: ERROR: ningun caption contiene 'nif'  gt=(525,1399) pred=(None,None) error=Nonepx
02_alta_cliente.png/anadir_button: ERROR: ningun caption contiene 'anadir'  gt=(475,1483) pred=(None,None) error=Nonepx


03_popup_validacion.png/popup_aceptar_button: ERROR: ningun caption contiene 'aceptar'  gt=(1397,888) pred=(None,None) error=Nonepx

Resumen omniparser: 0/7 aciertos (error < 20px)
error medio: nanpx
tiempo medio (deteccion+caption por imagen): 0.14s
capturas con circulos guardadas en: vision_evaluation\omniparser/


,imagen,target,gt_x,gt_y,pred_x,pred_y,error_px,tiempo_s,estado
0,01_login.png,usuario_field,272,138,None,None,None,0.149276,ERROR: ningun caption contiene 'usuario'
1,01_login.png,contraseña_field,272,188,None,None,None,0.149276,ERROR: ningun caption contiene 'contras'
2,01_login.png,entrar_button,208,274,None,None,None,0.149276,ERROR: ningun caption contiene 'entrar'
3,02_alta_cliente.png,nombre_field,219,1399,None,None,None,0.138420,ERROR: ningun caption contiene 'nombre'
4,02_alta_cliente.png,nif_field,525,1399,None,None,None,0.138420,ERROR: ningun caption contiene 'nif'
5,02_alta_cliente.png,anadir_button,475,1483,None,None,None,0.138420,ERROR: ningun caption contiene 'anadir'
6,03_popup_validacion.png,popup_aceptar_button,1397,888,None,None,None,0.141256,ERROR: ningun caption contiene 'aceptar'


## 4. Molmo (Hugging Face `transformers`)

⚠️ **No ejecutada por Claude.** `allenai/Molmo-7B-D-0924`, ~16 GB en bf16. Ejecuta esta celda tu cuando quieras descargarlo y cargarlo en la GPU.

Molmo tiene una capacidad nativa de "pointing": responde con `<point x="..." y="...">` usando **porcentajes (0-100)** del ancho/alto de la imagen, no pixeles — lo confirmaremos viendo la salida real aqui.

In [6]:
_molmo_model = None
_molmo_processor = None


def _load_molmo():
    global _molmo_model, _molmo_processor
    if _molmo_model is None:
        from transformers import AutoModelForCausalLM, AutoProcessor

        model_id = "allenai/Molmo-7B-D-0924"
        _molmo_processor = AutoProcessor.from_pretrained(
            model_id, trust_remote_code=True, torch_dtype="auto", device_map="auto"
        )
        _molmo_model = AutoModelForCausalLM.from_pretrained(
            model_id, trust_remote_code=True, torch_dtype="auto", device_map="auto"
        )
    return _molmo_model, _molmo_processor


def ask_molmo(image_name: str, target_desc: str) -> dict:
    from transformers import GenerationConfig

    model, processor = _load_molmo()
    image = Image.open(GT_DIR / image_name).convert("RGB")
    prompt = f"point to {target_desc}"

    inputs = processor.process(images=[image], text=prompt)
    inputs = {k: v.to(model.device).unsqueeze(0) for k, v in inputs.items()}

    t0 = time.perf_counter()
    output = model.generate_from_batch(
        inputs,
        GenerationConfig(max_new_tokens=200, stop_strings="<|endoftext|>"),
        tokenizer=processor.tokenizer,
    )
    elapsed = time.perf_counter() - t0

    generated_tokens = output[0, inputs["input_ids"].size(1):]
    raw = processor.tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return {"raw": raw, "elapsed_s": elapsed}


In [ ]:
# prueba rapida con un solo target: dispara la descarga de Molmo (~16 GB)
# la primera vez que se ejecuta, luego queda cacheado en disco
resultado = ask_molmo(TARGETS[2][0], TARGETS[2][2])
print(f"tiempo: {resultado['elapsed_s']:.1f}s\n")
print(resultado["raw"])


## 5. UI-TARS (Hugging Face `transformers`)

⚠️ **No ejecutada por Claude.** `bytedance-research/UI-TARS-7B-DPO`, ~16 GB en bf16, misma arquitectura que Qwen2-VL. Ejecuta esta celda tu cuando quieras descargarlo.

Esta pensado especificamente para agentes de GUI: espera que le pidas una accion (`click`) y responde con su propio formato de accion, del que sacaremos las coordenadas.

In [7]:
_uitars_model = None
_uitars_processor = None


def _load_uitars():
    global _uitars_model, _uitars_processor
    if _uitars_model is None:
        from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

        model_id = "ByteDance-Seed/UI-TARS-7B-DPO"
        _uitars_processor = AutoProcessor.from_pretrained(model_id)
        _uitars_model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_id, torch_dtype=torch.bfloat16, device_map="auto"
        )
    return _uitars_model, _uitars_processor


def ask_uitars(image_name: str, target_desc: str) -> dict:
    model, processor = _load_uitars()
    image = Image.open(GT_DIR / image_name).convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {
                    "type": "text",
                    "text": f"Click on {target_desc}. Respond with click(start_box='(x,y)').",
                },
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

    t0 = time.perf_counter()
    output_ids = model.generate(**inputs, max_new_tokens=128)
    elapsed = time.perf_counter() - t0

    raw = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )[0]
    return {"raw": raw, "elapsed_s": elapsed}


In [ ]:
# prueba rapida con un solo target: dispara la descarga de UI-TARS (~16 GB)
resultado = ask_uitars(TARGETS[2][0], TARGETS[2][2])
print(f"tiempo: {resultado['elapsed_s']:.1f}s\n")
print(resultado["raw"])
